# 36 · 缓存与延迟优化

> RAG 慢在哪？——**检索前**的 embedding、**生成**时的长 prompt + LLM。缓存是性价比最高的提速手段。

**本文件覆盖知识点**：Cache / Semantic Cache / Latency / Response Time / TTFT / Throughput / Cold Start / 优化清单

## 1. 延迟去哪了？

```text
总耗时 ≈ 请求路由 + 问题Embedding + 向量检索 + 重排 + LLM生成
                                                      └──── 大头，TTFT/首字与生成长度成正比
```

| 术语 | 含义 |
|------|------|
| **TTFT** | 首字返回时间（首 token 延迟） |
| **Throughput** | 每秒处理的请求/词元数（吞吐） |
| **Cold Start** | 冷启动：索引首次加载/进程首次拉起时的额外耗时 |

> 体感延迟 = TTFT + 生成时长；聊天机器人尤其看重 TTFT。

In [ ]:
# 语义缓存(Semantic Cache)：相似问题命中缓存，跳过 LLM
import numpy as np

class SemanticCache:
    def __init__(self, thr=0.93):
        self.items, self.thr = [], thr   # (向量, 答案)

    def embed(self, s):
        """教学用哈希向量；生产换成 text-embedding-v3"""
        v = np.zeros(16); [np.add.at(v, ord(c) % 16, 1) for c in s]; return v / (np.linalg.norm(v)+1e-9)

    def get(self, q):
        e = self.embed(q)
        best = max(((np.dot(e, i[0]), i[1]) for i in self.items), default=(0, None))
        return best[1] if best[0] >= self.thr else None

    def put(self, q, a):
        self.items.append((self.embed(q), a))

cache = SemanticCache()
cache.put('怎么申请报销', '登录 OA → 上传发票 → 审批')
print('相似问题命中缓存:', cache.get('报销怎么申请'))
print('无关问题未命中  :', cache.get('今天天气'))

## 2. 其它优化清单

### 检索侧
- 向量维度/量化（PQ/标量量化）降内存与延迟；
- 用 HNSW/ANN 代替暴力检索；适度减小候选集再重排；
- 只对必要字段做 embedding，控制索引体积。

### 生成侧（往往收益最大）
- 精简上下文（重排取 top-k 而非全塞）；
- 精简 prompt、限制生成长度、用更快的模型做简单任务；
- **流式输出**先给用户首字（TTFT 感知）。

### 系统侧
- **精确缓存**（同一问题原文命中，key=问题哈希）；
- **语义缓存**（向量相似命中，见上代码）；
- 预加载索引/模型（消除 Cold Start）、按需缩放副本。

## 小结

- 延迟大头是 **LLM 生成**，其次 embedding/检索；
- 缓存：先做**精确缓存**（零成本），再做**语义缓存**；
- 优化顺序建议：切上下文 → 缓存 → 索引加速 → 模型/流式。